# 03 — Keyword Co-Occurrence Network

Author keyword extraction, normalization, co-occurrence matrix, Louvain community detection, static network plot (matplotlib), and interactive pyvis network.

**Requires:** `data/processed/corpus_clean.parquet` (run NB00 first)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PROCESSED_DIR = "../data/processed/"
OUTPUTS_DIR   = "../outputs/"
MIN_KW_FREQ   = 3    # minimum keyword frequency to include in network
MIN_COOC      = 2    # minimum co-occurrence edge weight
TOP_KW_BAR    = 30   # top N keywords for bar chart
MAX_STATIC_NODES = 80  # max nodes in static PNG

In [ ]:
# ── LOAD ──────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import networkx as nx
from IPython.display import display, IFrame

from scripts.utils import (
    load_processed, split_multivalued, normalize_keywords,
    build_cooccurrence_matrix, freq_table, save_figure,
)

df = load_processed("corpus_clean")
N  = len(df)
N_with_kw = df["author_keywords"].notna().sum()
print(f"Corpus: N={N}  |  Records with author keywords: {N_with_kw} ({100*N_with_kw/N:.1f}%)")

## 1 · Keyword Extraction & Normalization

In [ ]:
# Explode author keywords (semicolon-separated)
kw_long = split_multivalued(df[df["author_keywords"].notna()], "author_keywords", sep=";")
kw_long["value"] = normalize_keywords(kw_long["value"])
kw_long = kw_long[kw_long["value"].str.len() > 0]

print(f"Total keyword occurrences: {len(kw_long):,}")
print(f"Unique keywords (before normalization): estimated higher")
print(f"Unique keywords (after normalization):  {kw_long['value'].nunique():,}")

kw_freq_all = freq_table(kw_long["value"])
print(f"\nTop 20 keywords:")
display(kw_freq_all.head(20))

## 2 · Top Keywords Bar Chart

In [ ]:
kw_top = kw_freq_all.head(TOP_KW_BAR)

fig = go.Figure(go.Bar(
    x=kw_top["N"],
    y=kw_top["value"],
    orientation="h",
    text=[f"N={n}" for n in kw_top["N"]],
    textposition="outside",
    marker_color="#1565C0",
))
fig.update_layout(
    title=f"Top {TOP_KW_BAR} Author Keywords (from {N_with_kw} records with keywords)",
    xaxis=dict(title="Frequency", range=[0, kw_top.N.max() * 1.3]),
    yaxis=dict(autorange="reversed"),
    height=max(400, TOP_KW_BAR * 22),
    margin=dict(l=200),
)
fig.show()
save_figure(fig, "03_top_keywords_bar")

## 3 · Co-Occurrence Matrix

In [ ]:
# Filter keywords by minimum frequency
freq_map = kw_freq_all.set_index("value")["N"].to_dict()
kw_long_filtered = kw_long[kw_long["value"].map(freq_map) >= MIN_KW_FREQ]

# Build per-record keyword lists
kw_per_record = (
    kw_long_filtered
    .groupby("Record_ID")["value"]
    .apply(list)
    .tolist()
)

cooc = build_cooccurrence_matrix(kw_per_record, min_cooc=MIN_COOC)
print(f"Keywords with freq ≥ {MIN_KW_FREQ}: {kw_long_filtered['value'].nunique()}")
print(f"Co-occurrence pairs with weight ≥ {MIN_COOC}: {len(cooc)}")

# Export
cooc_df = pd.DataFrame([(a, b, w) for (a, b), w in cooc.items()], columns=["kw_a", "kw_b", "weight"])
cooc_df.to_parquet(PROCESSED_DIR + "kw_cooccurrence.parquet", index=False)

kw_freq_filtered = kw_freq_all[kw_freq_all["value"].map(freq_map) >= MIN_KW_FREQ]
kw_freq_filtered.to_parquet(PROCESSED_DIR + "kw_freq.parquet", index=False)
print("✓ kw_cooccurrence.parquet and kw_freq.parquet exported")

## 4 · Network Construction & Community Detection

In [ ]:
G = nx.Graph()

# Nodes
for _, row in kw_freq_filtered.iterrows():
    G.add_node(row["value"], freq=row["N"])

# Edges
for (a, b), w in cooc.items():
    if G.has_node(a) and G.has_node(b):
        G.add_edge(a, b, weight=w)

# Remove isolated nodes
isolated = list(nx.isolates(G))
G.remove_nodes_from(isolated)
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (removed {len(isolated)} isolates)")

# Louvain community detection
communities = nx.community.louvain_communities(G, seed=42, weight="weight")
communities = sorted(communities, key=len, reverse=True)
comm_map = {}
for comm_id, nodes in enumerate(communities):
    for node in nodes:
        comm_map[node] = comm_id

print(f"Communities detected: {len(communities)}")
for i, comm in enumerate(communities[:6]):
    top_kws = sorted(comm, key=lambda n: freq_map.get(n, 0), reverse=True)[:5]
    print(f"  C{i} (N={len(comm)}): {', '.join(top_kws)}")

# Layout
pos = nx.spring_layout(G, seed=42, weight="weight", k=0.8)

## 5 · Static Network Visualization

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Take top MAX_STATIC_NODES by degree
top_nodes = sorted(G.nodes(), key=lambda n: G.degree(n, weight="weight"), reverse=True)[:MAX_STATIC_NODES]
H = G.subgraph(top_nodes)

n_comms = len(communities)
cmap    = cm.get_cmap("tab20", n_comms)
node_colors_static = [mcolors.to_hex(cmap(comm_map.get(n, 0))) for n in H.nodes()]

# Node sizes proportional to freq
max_freq = max(freq_map.values())
node_sizes = [300 + 1200 * (freq_map.get(n, 1) / max_freq) for n in H.nodes()]

# Edge widths
max_w  = max(d["weight"] for _, _, d in H.edges(data=True)) if H.number_of_edges() > 0 else 1
edge_widths = [0.3 + 3 * H[u][v]["weight"] / max_w for u, v in H.edges()]

# Labels only for high-frequency nodes
label_threshold = max(5, int(max_freq * 0.15))
labels = {n: n for n in H.nodes() if freq_map.get(n, 0) >= label_threshold}

pos_sub = {n: pos[n] for n in H.nodes()}

fig_static, ax = plt.subplots(figsize=(14, 11), dpi=150)
nx.draw_networkx_edges(H, pos_sub, ax=ax, width=edge_widths, alpha=0.4, edge_color="#90A4AE")
nx.draw_networkx_nodes(H, pos_sub, ax=ax, node_color=node_colors_static, node_size=node_sizes, alpha=0.9)
nx.draw_networkx_labels(H, pos_sub, labels, ax=ax, font_size=7, font_color="#212121", font_weight="bold")

# Community legend
legend_patches = []
for i, comm in enumerate(communities[:min(8, n_comms)]):
    top3 = sorted(comm, key=lambda n: freq_map.get(n, 0), reverse=True)[:3]
    label_text = f"C{i}: {', '.join(top3)}"
    legend_patches.append(mpatches.Patch(color=mcolors.to_hex(cmap(i)), label=label_text))

ax.legend(handles=legend_patches, loc="upper left", fontsize=7, framealpha=0.9)
ax.set_title(f"Author Keyword Co-Occurrence Network (top {len(H.nodes())} nodes; communities by Louvain)",
             fontsize=13, pad=12)
ax.axis("off")
fig_static.tight_layout()
fig_static.savefig(OUTPUTS_DIR + "03_keyword_network_static.svg", bbox_inches="tight")
fig_static.savefig(OUTPUTS_DIR + "03_keyword_network_static.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ 03_keyword_network_static.svg + .png saved")

## 6 · Interactive Network (pyvis)

In [ ]:
from pyvis.network import Network
import json

# ── Temporal color preparation (shared with Section 8) ────────────────────
year_col = "year" if "year" in df.columns else "Year"
_kw_yr = kw_long_filtered.copy()
_kw_yr["year"] = _kw_yr["Record_ID"].map(df.set_index("Record_ID")[year_col].to_dict())
mean_year_map = _kw_yr.groupby("value")["year"].mean().to_dict()

YEAR_MIN, YEAR_MAX = 2000, 2026
norm_t = mcolors.Normalize(vmin=YEAR_MIN, vmax=YEAR_MAX)
cmap_t = cm.get_cmap("plasma")

# ── Pre-build both color dicts for the toggle ──────────────────────────────
community_colors_js = {
    node: mcolors.to_hex(cmap(comm_map.get(node, 0) % n_comms))
    for node in G.nodes()
}
temporal_colors_js = {
    node: mcolors.to_hex(cmap_t(norm_t(mean_year_map.get(node, (YEAR_MIN + YEAR_MAX) / 2))))
    for node in G.nodes()
}

# ── Build legend HTML ──────────────────────────────────────────────────────
# Community legend: one row per community (top 3 keywords)
_comm_rows = []
for i, comm in enumerate(communities):
    nodes_in_g = [n for n in comm if G.has_node(n)]
    if not nodes_in_g:
        continue
    top3   = sorted(nodes_in_g, key=lambda n: freq_map.get(n, 0), reverse=True)[:3]
    color  = mcolors.to_hex(cmap(i % n_comms))
    _comm_rows.append(
        f'<div style="margin:2px 0">'
        f'<span style="display:inline-block;width:12px;height:12px;'
        f'background:{color};border-radius:2px;margin-right:6px;vertical-align:middle"></span>'
        f'<b>C{i}</b>: {", ".join(top3)}</div>'
    )
_comm_legend_html = "\n".join(_comm_rows)

# Temporal legend: plasma gradient bar + year axis
_n_stops = 8
_stops   = [mcolors.to_hex(cmap_t(i / (_n_stops - 1))) for i in range(_n_stops)]
_grad    = ", ".join(_stops)
_tick_years = [YEAR_MIN + round(i * (YEAR_MAX - YEAR_MIN) / 4) for i in range(5)]
_ticks_html = "".join(
    f'<span style="position:absolute;left:{round(i*25)}%;transform:translateX(-50%)">{y}</span>'
    for i, y in enumerate(_tick_years)
)

# ── Build pyvis network ────────────────────────────────────────────────────
net = Network(height="750px", width="100%", bgcolor="#fafafa", font_color="#212121",
              directed=False, notebook=True, cdn_resources="in_line")

def scale_size(freq, min_s=8, max_s=40):
    return min_s + (max_s - min_s) * (freq / max_freq)

for node in G.nodes():
    freq = freq_map.get(node, 1)
    c_id = comm_map.get(node, 0)
    my   = mean_year_map.get(node, (YEAR_MIN + YEAR_MAX) / 2)
    net.add_node(
        node, label=node, size=scale_size(freq),
        color=community_colors_js[node],
        title=(
            f"{node}<br>freq={freq}<br>community=C{c_id}"
            f"<br>mean_year={my:.1f}<br>degree={G.degree(node)}"
        ),
    )

for u, v, data in G.edges(data=True):
    net.add_edge(u, v, value=data["weight"], title=f"co-occurrence={data['weight']}")

net.set_options(json.dumps({
    "physics": {"enabled": True, "barnesHut": {"gravitationalConstant": -8000, "springLength": 120}},
    "interaction": {"hover": True, "tooltipDelay": 100},
}))

html_path = OUTPUTS_DIR + "03_keyword_network.html"
net.save_graph(html_path)

# ── Inject toggle + legend via HTML post-processing ───────────────────────
_inject = f"""
<style>
  #ctrl-panel {{
    position:fixed; top:12px; right:12px; z-index:9999;
    background:rgba(255,255,255,0.93); border:1px solid #ccc;
    border-radius:6px; padding:10px 14px; min-width:220px;
    font-family:sans-serif; font-size:12px;
    box-shadow:0 2px 8px rgba(0,0,0,0.15);
  }}
  #ctrl-panel .toggle-row {{ margin-bottom:8px; font-size:13px; }}
  #ctrl-panel button {{
    margin:0 3px; padding:3px 10px; border-radius:4px;
    border:1px solid #999; cursor:pointer; font-size:12px; transition:0.15s;
  }}
  #ctrl-panel button.active {{ background:#1565C0; color:white; border-color:#1565C0; }}
  #legend-comm {{ line-height:1.6; }}
  #legend-temp {{ display:none; }}
  .grad-bar {{
    height:14px; border-radius:3px;
    background:linear-gradient(to right, {_grad});
    margin:6px 0 2px;
  }}
  .grad-ticks {{ position:relative; height:16px; font-size:11px; color:#555; }}
</style>
<div id="ctrl-panel">
  <div class="toggle-row">
    Color by:
    <button id="btn-comm" class="active" onclick="setColorMode('community')">Communities</button>
    <button id="btn-temp" onclick="setColorMode('temporal')">Temporal</button>
  </div>
  <div id="legend-comm">
    {_comm_legend_html}
  </div>
  <div id="legend-temp">
    <div class="grad-bar"></div>
    <div class="grad-ticks">{_ticks_html}</div>
  </div>
</div>
<script>
var _communityColors = {json.dumps(community_colors_js)};
var _temporalColors  = {json.dumps(temporal_colors_js)};
function setColorMode(mode) {{
  document.getElementById('btn-comm').className = (mode==='community') ? 'active' : '';
  document.getElementById('btn-temp').className = (mode==='temporal')  ? 'active' : '';
  document.getElementById('legend-comm').style.display = (mode==='community') ? 'block' : 'none';
  document.getElementById('legend-temp').style.display = (mode==='temporal')  ? 'block' : 'none';
  var cm = (mode==='community') ? _communityColors : _temporalColors;
  nodes.update(Object.keys(cm).map(function(id) {{
    return {{id:id, color:{{background:cm[id], border:cm[id],
             highlight:{{background:cm[id], border:'#fff'}}}}}};
  }}));
}}
</script>
"""

with open(html_path, "r", encoding="utf-8") as f:
    _html = f.read()
_html = _html.replace("</body>", _inject + "\n</body>")
with open(html_path, "w", encoding="utf-8") as f:
    f.write(_html)

print(f"✓ Interactive network (Communities / Temporal toggle + legend) saved: {html_path}")
IFrame(html_path, width=950, height=760)

## 7 · Community Summary Table

In [ ]:
comm_summary_rows = []
for i, comm in enumerate(communities):
    nodes_in_g = [n for n in comm if G.has_node(n)]
    if not nodes_in_g:
        continue
    top5 = sorted(nodes_in_g, key=lambda n: freq_map.get(n, 0), reverse=True)[:5]
    subH = G.subgraph(nodes_in_g)
    comm_summary_rows.append({
        "Community": f"C{i}",
        "Nodes": len(nodes_in_g),
        "Internal edges": subH.number_of_edges(),
        "Top 5 keywords": ", ".join(top5),
    })

comm_df = pd.DataFrame(comm_summary_rows)
display(comm_df)

# Export community membership
kw_comm_export = pd.DataFrame([
    {"keyword": n, "freq": freq_map.get(n, 0), "community_id": comm_map.get(n, -1), "degree": G.degree(n)}
    for n in G.nodes()
])
kw_comm_export.to_parquet(PROCESSED_DIR + "kw_communities.parquet", index=False)
print("✓ kw_communities.parquet exported")

## 8 · Temporal Gradient Network (static)

Same spring-layout positions as Section 5; nodes coloured by **mean publication year** of papers they appear in (plasma gradient: dark-purple = early, yellow = recent).  
The interactive network in Section 6 includes the same coloring via the **Communities / Temporal** toggle button.

In [ ]:
# ── 8: Static temporal matplotlib — same layout as Section 5 ──────────────
# mean_year_map, cmap_t, norm_t, YEAR_MIN, YEAR_MAX computed in Section 6

node_colors_t = [
    mcolors.to_hex(cmap_t(norm_t(mean_year_map.get(n, (YEAR_MIN + YEAR_MAX) / 2))))
    for n in H.nodes()
]

fig_t, ax_t = plt.subplots(figsize=(14, 11), dpi=150)
nx.draw_networkx_edges(H, pos_sub, ax=ax_t, width=edge_widths, alpha=0.4, edge_color="#90A4AE")
nx.draw_networkx_nodes(H, pos_sub, ax=ax_t, node_color=node_colors_t,
                       node_size=node_sizes, alpha=0.9)
nx.draw_networkx_labels(H, pos_sub, labels, ax=ax_t,
                        font_size=7, font_color="#212121", font_weight="bold")

sm = cm.ScalarMappable(cmap=cmap_t, norm=norm_t)
sm.set_array([])
cbar = fig_t.colorbar(sm, ax=ax_t, fraction=0.03, pad=0.02)
cbar.set_label("Mean publication year", fontsize=10)
cbar.set_ticks([2000, 2006, 2012, 2019, 2026])

ax_t.set_title(
    f"Keyword Network — Temporal Gradient (top {len(H.nodes())} nodes; node size ∝ frequency)",
    fontsize=13, pad=12,
)
ax_t.axis("off")
fig_t.tight_layout()
fig_t.savefig(OUTPUTS_DIR + "03_keyword_network_temporal_static.svg", bbox_inches="tight")
fig_t.savefig(OUTPUTS_DIR + "03_keyword_network_temporal_static.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ 03_keyword_network_temporal_static.svg + .png saved")